1. Preparing the Data for SVM
SVM is a geometric algorithm that relies on calculating distances between data points. Because of this, feature scaling is mandatory. If one variable has a massive range (like income), it will completely dominate the distance calculations, rendering other features useless.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Load dataset
df = pd.read_csv("medical_insurance.csv")

continuous_features = ["age", "income", "bmi", "visits_last_year", "risk_score", "claims_count"]
categorical_features = ["sex", "region", "urban_rural", "smoker", "plan_type"]
target_raw = "annual_medical_cost"

# Clean missing values
df_clean = df.dropna(subset=continuous_features + categorical_features + [target_raw])

# Define binary classification target: 1 if high cost (top 25%), else 0
cost_threshold = df_clean[target_raw].quantile(0.75)
df_clean["is_high_cost"] = (df_clean[target_raw] > cost_threshold).astype(int)
y = df_clean["is_high_cost"]

# Encode categorical variables
X_raw = df_clean[continuous_features + categorical_features]
X_encoded = pd.get_dummies(X_raw, columns=categorical_features, drop_first=True).astype(float)

# Split into Train and Test sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Standardize continuous features (Crucial for SVM distance calculations!)
scaler = StandardScaler()
X_train_scaled = X_train_raw.copy()
X_test_scaled = X_test_raw.copy()

X_train_scaled[continuous_features] = scaler.fit_transform(X_train_raw[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test_raw[continuous_features])

print(f"✅ Data standardized. Training shape: {X_train_scaled.shape}")

✅ Data standardized. Training shape: (80000, 19)


2. Adjusting Regularization ($C$) and Identifying Support VectorsThe parameter $C$ balances margin size against training accuracy.

- Small $C$: Maximizes the margin width by allowing some training points to be misclassified or violate the margin (Soft Margin). This prioritizes generalization over training accuracy.
- Large $C$: Penalizes misclassifications severely, forcing the margin to shrink to fit the training data perfectly (Hard Margin). This can lead to overfitting.

In [4]:
# Train an SVM with a soft margin (Small C) to observe support vectors
svm_model = SVC(kernel="linear", C=0.1, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Identify the support vectors
n_support_vectors = svm_model.n_support_
total_sv = np.sum(n_support_vectors)

print("\n=== SVM MARGIN AND SUPPORT VECTOR DIAGNOSTICS ===")
print(f"Number of Support Vectors per class (0 / 1): {n_support_vectors}")
print(f"Total data points acting as Support Vectors: {total_sv} out of {len(X_train_scaled)}")
print(f"Percentage of dataset anchoring the boundary: {(total_sv / len(X_train_scaled)) * 100:.2f}%")


=== SVM MARGIN AND SUPPORT VECTOR DIAGNOSTICS ===
Number of Support Vectors per class (0 / 1): [20451 19965]
Total data points acting as Support Vectors: 40416 out of 80000
Percentage of dataset anchoring the boundary: 50.52%


3. Applying the Kernel Trick (Linear vs. Polynomial vs. RBF)
When data is non-linearly separable in its raw state, we use Kernel Functions to map features into a higher-dimensional space where a clean linear boundary becomes possible.

In [5]:
# Initialize different kernel types
kernels = {
    "Linear Kernel": SVC(kernel="linear", C=1.0, random_state=42),
    "Polynomial Kernel (Degree 3)": SVC(kernel="poly", degree=3, C=1.0, random_state=42),
    "Gaussian RBF Kernel": SVC(kernel="rbf", gamma="scale", C=1.0, random_state=42)
}

print("\n=== COMPARING KERNEL FUNCTIONS ===")
for name, model in kernels.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, model.decision_function(X_test_scaled))
    print(f"• {name:28} | Test Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")


=== COMPARING KERNEL FUNCTIONS ===
• Linear Kernel                | Test Accuracy: 0.7482 | ROC-AUC: 0.5469
• Polynomial Kernel (Degree 3) | Test Accuracy: 0.7537 | ROC-AUC: 0.6441
• Gaussian RBF Kernel          | Test Accuracy: 0.7550 | ROC-AUC: 0.6176


4. Designing an Optimal Modeling StrategyThis decision architecture dynamically evaluates various combinations of kernels and regularization constraints ($C$) to locate the exact model configuration that optimizes performance without overfitting.

In [ ]:
def select_optimal_svm_strategy(X_tr, X_te, y_tr, y_te):
    """
    Grid-searches combinations of structural Kernels and Regularization paths
    to choose the optimal SVM strategy based on validation profiles.
    """
    best_score = 0
    best_config = {}
    strategy_logs = []
    
    # Grid of parameters to test
    candidate_kernels = ["linear", "poly", "rbf"]
    candidate_c_values = [0.01, 1.0, 100.0]
    
    for k in candidate_kernels:
        for c in candidate_c_values:
            # Instantiate and fit
            clf = SVC(kernel=k, C=c, random_state=42)
            clf.fit(X_tr, y_tr)
            
            # Score using area under the ROC curve
            test_auc = roc_auc_score(y_te, clf.decision_function(X_te))
            train_auc = roc_auc_score(y_tr, clf.decision_function(X_tr))
            
            # Count supporting matrix points
            sv_count = np.sum(clf.n_support_)
            
            strategy_logs.append({
                "Kernel": k, "C": c, "Test_AUC": test_auc, 
                "Train_AUC": train_auc, "SV_Count": sv_count
            })
            
            if test_auc > best_score:
                best_score = test_auc
                best_config = {"kernel": k, "C": c, "model": clf, "sv_count": sv_count}
                
    # Output analysis report
    print("\n================ SYSTEM DESIGN SELECTION STRATEGY ================")
    log_df = pd.DataFrame(strategy_logs)
    print(log_df.to_string(index=False))
    
    print(f"\n🚀 STRATEGIC RECOMMENDATION: Deploy SVM with a **{best_config['kernel'].upper()}** kernel and **C = {best_config['C']}**.")
    print(f"• Final Selection Test AUC: {best_score:.4f}")
    print(f"• Final Selected Support Vector Nodes: {best_config['sv_count']} bounds elements.")
    
    # Strategic reasoning breakdown
    if best_config["kernel"] == "linear":
        print("REASONING: Linear separability dominates the feature landscape. Complex multi-dimensional mappings are unnecessary.")
    else:
        print(f"REASONING: Significant non-linear interactions exist among medical risks. The {best_config['kernel']} kernel successfully uncovers these relationships.")
        
    if best_config["C"] == 100.0:
        print("REASONING: High boundary rigidity (large C) maximizes classification margins without introducing train-set overfitting variance.")
    elif best_config["C"] == 0.01:
        print("REASONING: A soft margin (small C) protects the pipeline against noise and outliers by expanding the decision cushion.")

# Run Selection Strategy
select_optimal_svm_strategy(X_train_scaled, X_test_scaled, y_train, y_test)